 # Stochastic Methods in Finance

## Importing packages

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

## Importing and inspecting the dataset

In [2]:
data = pd.read_csv("./HistoricalData_1746127004374.csv")
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2515 entries, 0 to 2514
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Date        2515 non-null   object
 1   Close/Last  2515 non-null   object
 2   Volume      2515 non-null   int64 
 3   Open        2515 non-null   object
 4   High        2515 non-null   object
 5   Low         2515 non-null   object
dtypes: int64(1), object(5)
memory usage: 118.0+ KB


In [3]:
data.head()

,Date,Close/Last,Volume,Open,High,Low
0,04/30/2025,$395.26,36461080,$390.30,$396.66,$384.44
1,04/29/2025,$394.04,14973980,$391.30,$395.10,$390.38
2,04/28/2025,$391.16,16579430,$391.955,$392.74,$386.638
3,04/25/2025,$391.85,18973170,$387.00,$392.16,$384.60
4,04/24/2025,$387.30,22232290,$375.695,$388.45,$375.19


In [4]:
data.tail()

,Date,Close/Last,Volume,Open,High,Low
2510,05/07/2015,$46.70,32940820,$46.27,$47.085,$46.16
2511,05/06/2015,$46.28,52416760,$47.57,$47.77,$46.02
2512,05/05/2015,$47.60,50364250,$47.82,$48.16,$47.31
2513,05/04/2015,$48.24,33986240,$48.37,$48.87,$48.18
2514,05/01/2015,$48.655,36432670,$48.58,$48.875,$48.40


## Preprocessing

In [5]:
# Converting the date column to datetime
data["Date"] = pd.to_datetime(data["Date"])
# Setting the date as the index
data.set_index(["Date"], inplace=True)

# We start from the last 5 years of data
data = data[data.index >= '2020-01-01']

# Setting the variables as numeric
# Remove dollar signs and any other non-numeric characters, then convert to numeric
data['Close/Last'] = data['Close/Last'].replace({'\$': '', ',': ''}, regex=True).astype(float)
data['Open'] = data['Open'].replace({'\$': '', ',': ''}, regex=True).astype(float)
data['High'] = data['High'].replace({'\$': '', ',': ''}, regex=True).astype(float)
data['Low'] = data['Low'].replace({'\$': '', ',': ''}, regex=True).astype(float)
data['Volume'] = data['Volume'].replace({',': ''}, regex=True).astype(int)

# Check the types of the columns
print(data.dtypes)

Close/Last    float64
Volume          int64
Open          float64
High          float64
Low           float64
dtype: object


<>:11: SyntaxWarning: invalid escape sequence '\$'
<>:12: SyntaxWarning: invalid escape sequence '\$'
<>:13: SyntaxWarning: invalid escape sequence '\$'
<>:14: SyntaxWarning: invalid escape sequence '\$'
<>:11: SyntaxWarning: invalid escape sequence '\$'
<>:12: SyntaxWarning: invalid escape sequence '\$'
<>:13: SyntaxWarning: invalid escape sequence '\$'
<>:14: SyntaxWarning: invalid escape sequence '\$'
/var/folders/wn/nf_gxvtd0zsfbsgzc8y_qpn40000gn/T/ipykernel_15344/1173805684.py:11: SyntaxWarning: invalid escape sequence '\$'
  data['Close/Last'] = data['Close/Last'].replace({'\$': '', ',': ''}, regex=True).astype(float)
/var/folders/wn/nf_gxvtd0zsfbsgzc8y_qpn40000gn/T/ipykernel_15344/1173805684.py:12: SyntaxWarning: invalid escape sequence '\$'
  data['Open'] = data['Open'].replace({'\$': '', ',': ''}, regex=True).astype(float)
/var/folders/wn/nf_gxvtd0zsfbsgzc8y_qpn40000gn/T/ipykernel_15344/1173805684.py:13: SyntaxWarning: invalid escape sequence '\$'
  data['High'] = data['High']

In [31]:
data.head()

,Close/Last,Volume,Open,High,Low,Return
Date,,,,,,
2025-04-30,395.26,36461080,390.300,396.66,384.440,NaN
2025-04-29,394.04,14973980,391.300,395.10,390.380,-0.003087
2025-04-28,391.16,16579430,391.955,392.74,386.638,-0.007309
2025-04-25,391.85,18973170,387.000,392.16,384.600,0.001764
2025-04-24,387.30,22232290,375.695,388.45,375.190,-0.011612


In [30]:
data.tail()

,Close/Last,Volume,Open,High,Low,Return
Date,,,,,,
2020-01-08,160.09,27762030,158.93,160.800,157.9491,-0.012339
2020-01-07,157.58,21881740,159.32,159.670,157.3200,-0.015679
2020-01-06,159.03,20826700,157.08,159.100,156.5100,0.009202
2020-01-03,158.62,21121680,158.32,159.945,158.0600,-0.002578
2020-01-02,160.62,22634550,158.78,160.730,158.3300,0.012609


## Tasks

### 1. Creating the binomial tree

In [56]:
## Parameters
# Initial stock price S0 (price on April 28, 2025)
S_0 = data.loc['2025-04-28', 'Close/Last']  # Update with the correct date or your preferred method

# Parameters for the binomial model
n = 25             
T = 125 # 6 months
dt = T / n # Each time step corresponds to 5 days

# Calculate daily returns
data['Return'] = (data['Close/Last']).pct_change()

# Volatility & drift
sigma_daily = data['Return'].std() 
sigma_5days = sigma_daily * np.sqrt(dt)
r_annual    = 0.01
r_daily     = (1 + r_annual) ** (1/250) - 1
r_5days = (1 + r_daily) ** dt - 1 

# Up/down factors
u = np.exp(sigma_5days)  # up factor
d = np.exp(-sigma_5days) # down factor

In [51]:
## Binomial tree construction
# Initialize the binomial tree as a list of lists
tree = []

# Root node is S_0
tree.append([S_0])

# Fill the tree with prices
for i in range(1, n + 1):  # Loop through each level
    level = []
    for j in range(2 ** i):  # Each level has 2^i nodes
        # Calculate the price for each node
        up_moves = bin(j).count('1')  # Count of '1's in binary representation gives up moves
        down_moves = i - up_moves
        price = S_0 * (u ** up_moves) * (d ** down_moves)
        level.append(price)
    tree.append(level)

# Number of terminal nodes
num_terminal_nodes = len(tree[-1])

# Print the number of terminal nodes
print(f"Number of terminal nodes: {num_terminal_nodes}")

Number of terminal nodes: 33554432


### 2. Average prices for each node

In [57]:
# Initialize a list to store average prices at terminal nodes
average_prices = []

# Loop through each terminal path (there are 2^n paths)
for j in range(2 ** n):
    # Determine the up/down path as a binary string
    path = bin(j)[2:].zfill(n)  # Pad to ensure length n
    prices = [S_0]  # Start with the initial price

    current_price = S_0
    for move in path:
        if move == '1':
            current_price *= u  # Up move
        else:
            current_price *= d  # Down move
        prices.append(current_price)

    # Compute the arithmetic average of the prices along the path
    avg_price = np.mean(prices)
    average_prices.append(avg_price)


### 3. Payoffs of the Asian option
##### Since we do not have a value for the strike price, we assume that the European Asian call option is at the money, i.e. the strike price is equal the initial stock price.

In [58]:
# Assume strike price is equal to initial price
K = S_0

# Compute payoffs for each terminal node (Asian call option)
payoffs = [max(avg_price - K, 0) for avg_price in average_prices]

# Verifying the number of payoffs at terminal nodes
print(f"Number of terminal payoffs: {len(payoffs)}")
print(f"Expected number of terminal nodes: {2 ** n}")
print("Match:", len(payoffs) == 2 ** n)


Number of terminal payoffs: 33554432
Expected number of terminal nodes: 33554432
Match: True


### 4. Risk-neutral probabilities

In [59]:
p_risk_neutral = (1 + r_5days - d) / (u - d)
q_risk_neutral = 1 - p_risk_neutral

### 5. Backward induction

In [60]:
asian_option_price = 0.0
for j in range(2**n):
    up_moves    = bin(j).count('1')
    down_moves  = n - up_moves
    path_prob   = (p_risk_neutral ** up_moves) * (q_risk_neutral ** down_moves)
    asian_option_price += payoffs[j] * path_prob

asian_option_price *= (1 + r_5days)**(-n)
print(f"Asian Call Option Price: {asian_option_price:.4f}")


Asian Call Option Price: 19.8022


### 6. Robustness check
#### To analyse the sensitivity of the option, we can change the risk-free rate and the volatility to understand how the price of the option changes

In [63]:
interest_rates = [0.02, 0.05, 0.10]  # annual r
volatilities   = [0.05, 0.10, 0.20]  # annual σ

results = {}

for r_annual in interest_rates:
    # per-day rate
    r_daily = (1 + r_annual)**(1/250) - 1
    # 5-day discrete drift
    r_5days = (1 + r_daily)**dt - 1

    for sigma_annual in volatilities:
        # convert to per-day vol
        sigma_daily = sigma_annual / np.sqrt(250)
        # 5-day vol
        sigma_5days = sigma_daily * np.sqrt(dt)

        # per-step up/down
        u = np.exp(sigma_5days)
        d = np.exp(-sigma_5days)

        # risk-neutral probs over 5 days
        p_risk_neutral = ((1 + r_5days) - d) / (u - d)
        q_risk_neutral = 1 - p_risk_neutral

        # price by full enumeration
        price = 0.0
        for j in range(2**n):
            bits     = bin(j)[2:].zfill(n)
            S        = S_0
            ups      = 0
            acc      = S_0

            for b in bits:
                if b == '1':
                    S   *= u
                    ups += 1
                else:
                    S   *= d
                acc += S

            avg_price = acc / (n + 1)
            payoff    = max(avg_price - K, 0)
            downs     = n - ups

            price   += payoff * (p_risk_neutral**ups) * (q_risk_neutral**downs)

        # discount back 125 days via discrete 5-day compounding
        price *= (1 + r_5days)**(-n)

        results[(r_annual, sigma_annual)] = price
        print(f"r={r_annual:.1%}, σ={sigma_annual:.1%} → Asian call ≃ {price:.4f}")


r=2.0%, σ=5.0% → Asian call ≃ 4.2052
r=2.0%, σ=10.0% → Asian call ≃ 7.3119
r=2.0%, σ=20.0% → Asian call ≃ 13.5887
r=5.0%, σ=5.0% → Asian call ≃ 6.0034
r=5.0%, σ=10.0% → Asian call ≃ 8.8633
r=5.0%, σ=20.0% → Asian call ≃ 14.9667
r=10.0%, σ=5.0% → Asian call ≃ 9.4629
r=10.0%, σ=10.0% → Asian call ≃ 11.6680
r=10.0%, σ=20.0% → Asian call ≃ 17.3180


The effect of the interest rate is practically nonexistent since we are converting the annual interest rate to a daily rate and the period taken into consideration is only 25 days, which is too short to observe any change. 

### 7. Price approximation
#### Here, we use the normal approximation of the binomial distribution to compute the price of the Asian call option. The payoffs depend on the average prices and, therefore, unlikely a vanilla option, we cannot just change the mean and the variance approximating. Here, we will moment-match the dsitribution of the average.  

In [66]:
for n in [500, 1000]:
        K = S_0
        mean_A = np.mean(S_0 * np.exp(r_daily * T))
        var_A = (sigma_daily ** 2 / (n + 1)) \
                * (1 - np.exp(-2 * r_daily * T)) \
                / (2 * r_daily)
        sigma_A = np.sqrt(var_A)
        d = (mean_A - K) / sigma_A
        payoff = (mean_A - K)*norm.cdf(d) + sigma_A*norm.pdf(d)

price_norm_approx = np.exp(-r_daily * T) * payoff
print(f"Asian call (Normal approx): {price_norm_approx:.4f}")

Asian call (Normal approx): 18.2070


Since we only have 25 periods for the binomial model, which are not sufficient for the Central Limit Theorem to "kick-in" when normally approximating the binomial distribution, we can use a new range for n to reflect the 4 years of daily data we are using for this project. 